In [ ]:
import pandas as pd
import re, string
import nltk
from google.colab import files

In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:


columns = ['target', 'id', 'date', 'flag', 'user', 'text']

df = pd.read_csv(
    "training.1600000.processed.noemoticon.csv",
    encoding="latin-1",
    names=columns,
    header=None,
    engine="python",
    on_bad_lines="skip"
)

print(df.shape)
df.head()

(1600000, 6)


,target,id,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [ ]:
TEXT_COL = "text"
df = df[["target", "text"]]
df = df.groupby("target").sample(10000, random_state=42).reset_index(drop=True)
print(df.shape)
print(df["target"].value_counts())
df.head()

(20000, 2)
target
0    10000
4    10000
Name: count, dtype: int64


,target,text
0,0,@xnausikaax oh no! where did u order from? tha...
1,0,A great hard training weekend is over. a coup...
2,0,"Right, off to work Only 5 hours to go until I..."
3,0,I am craving for japanese food
4,0,Jean Michel Jarre concert tomorrow gotta work...


In [ ]:
from nltk.tokenize import sent_tokenize
df["sentences"] = df[TEXT_COL].apply(lambda x: sent_tokenize(str(x)))
df[[TEXT_COL, "sentences"]].head()

,text,sentences
0,@xnausikaax oh no! where did u order from? tha...,"[@xnausikaax oh no!, where did u order from?, ..."
1,A great hard training weekend is over. a coup...,"[A great hard training weekend is over., a cou..."
2,"Right, off to work Only 5 hours to go until I...","[Right, off to work Only 5 hours to go until ..."
3,I am craving for japanese food,[I am craving for japanese food]
4,Jean Michel Jarre concert tomorrow gotta work...,[Jean Michel Jarre concert tomorrow gotta wor...


In [ ]:
def clean_tweet(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"&\w+;", "", text)
    return text

df["text_clean"] = df[TEXT_COL].apply(clean_tweet)
df[[TEXT_COL, "text_clean"]].head()

,text,text_clean
0,@xnausikaax oh no! where did u order from? tha...,oh no! where did u order from? that's horrible
1,A great hard training weekend is over. a coup...,a great hard training weekend is over. a coup...
2,"Right, off to work Only 5 hours to go until I...","right, off to work only 5 hours to go until i..."
3,I am craving for japanese food,i am craving for japanese food
4,Jean Michel Jarre concert tomorrow gotta work...,jean michel jarre concert tomorrow gotta work...


In [ ]:
from nltk.tokenize import word_tokenize
df["tokens"] = df["text_clean"].apply(word_tokenize)
df[["tokens"]].head()

,tokens
0,"[oh, no, !, where, did, u, order, from, ?, tha..."
1,"[a, great, hard, training, weekend, is, over, ..."
2,"[right, ,, off, to, work, only, 5, hours, to, ..."
3,"[i, am, craving, for, japanese, food]"
4,"[jean, michel, jarre, concert, tomorrow, got, ..."


In [ ]:
def clean_tokens(tokens):
    out = []
    for t in tokens:
        t = t.translate(str.maketrans("", "", string.punctuation))
        t = re.sub(r"\d+", "", t)
        if t:
            out.append(t)
    return out

df["tokens_no_punct"] = df["tokens"].apply(clean_tokens)
df[["tokens_no_punct"]].head()

,tokens_no_punct
0,"[oh, no, where, did, u, order, from, that, s, ..."
1,"[a, great, hard, training, weekend, is, over, ..."
2,"[right, off, to, work, only, hours, to, go, un..."
3,"[i, am, craving, for, japanese, food]"
4,"[jean, michel, jarre, concert, tomorrow, got, ..."


In [ ]:
from nltk.corpus import stopwords
stop_words = set(stopwords.words("english"))

df["tokens_no_stop"] = df["tokens_no_punct"].apply(
    lambda toks: [t for t in toks if t not in stop_words]
)
df[["tokens_no_stop"]].head()

,tokens_no_stop
0,"[oh, u, order, horrible]"
1,"[great, hard, training, weekend, couple, days,..."
2,"[right, work, hours, go, free, xd]"
3,"[craving, japanese, food]"
4,"[jean, michel, jarre, concert, tomorrow, got, ..."


In [ ]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

df["final_tokens"] = df["tokens_no_stop"].apply(
    lambda toks: [lemmatizer.lemmatize(t) for t in toks]
)

In [ ]:
df["clean_text"] = df["final_tokens"].apply(" ".join)

for i in range(3):
    print("BEFORE:", df[TEXT_COL].iloc[i])
    print("AFTER :", df["clean_text"].iloc[i])
    print("-"*60)

BEFORE: @xnausikaax oh no! where did u order from? that's horrible 
AFTER : oh u order horrible
------------------------------------------------------------
BEFORE: A great hard training weekend is over.  a couple days of rest and lets do it again!  Lots of computer time to put in now 
AFTER : great hard training weekend couple day rest let lot computer time put
------------------------------------------------------------
BEFORE: Right, off to work  Only 5 hours to go until I'm free xD
AFTER : right work hour go free xd
------------------------------------------------------------


In [ ]:
df.to_csv("preprocessed.csv", index=False)
files.download("preprocessed.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>